# 02o1: Extracting Data from NotePlan Files

This notebook demonstrates how to extract entities and relationships from NotePlan notes using the graph builder agent.

## Prerequisites

**⚠️ Important:** Before running this notebook, ensure you have:
- Completed [**00-import.ipynb**](./00-import.ipynb) for environment detection and Neo4j connection setup

All environment detection, Neo4j connection, and NotePlan directory configuration are handled in `00-import.ipynb`.

## Overview

This notebook reads your NotePlan notes and uses AI to identify important information like people, projects, tasks, and how they relate to each other. It's like having an assistant read through your notes and highlight the key connections.

**Alternative Approach:** If you prefer using LangChain's structured output capabilities instead of agents, see [**02o2-extracting-data-langchain.ipynb**](./02o2-extracting-data-langchain.ipynb) for an alternative extraction method.

We'll:
1. Load NotePlan files from appropriate path
2. Use graph builder agent to extract entities and relationships
3. Display extracted data (not storing yet - that's in the next notebook)


In [1]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
import asyncio
from knowledge_agents.agents.graph_builder_agent import run_graph_builder_agent

# NotePlan utilities
from notes.traversal import get_files_from_last_month
from notes.parser import read_noteplan_file
from notes.filter import should_skip_file

print("✅ Additional libraries imported")


✅ Added to path: /Users/omareid/Workspace/git/knowledge-agents/src
   Project root: /Users/omareid/Workspace/git/knowledge-agents
✅ Standard library imports loaded
✅ Repository components imported
🔑 Found NEO4J_PASSWORD env var (length: 5, preview: a***n)
⚠️  Auto-correcting: NEO4J_PASSWORD is 'admin' but should be 'admin123'
   Overriding to correct password. To fix permanently: export NEO4J_PASSWORD=admin123
🔍 Runtime detection: local
🔍 Environment variables: NEO4J_URI=bolt://host.docker.internal:7687, NEO4J_PASSWORD=set, LITELLM_PROXY_HOST=not set
✅ Final Settings values: neo4j_uri=bolt://localhost:7687, litellm_proxy_host=localhost
   Neo4j credentials: username=neo4j, password=******** (length: 8)
   💡 If auth fails, override password: settings = get_settings(neo4j_password='your_actual_password')
✅ Settings loaded - 💻 Mac (Local)
   Neo4j URI: bolt://localhost:7687
   Neo4j Database: knowledge
   Neo4j Username: neo4j
   Neo4j Password: ********
   LiteLLM Proxy Host: localhost
 

In [ ]:
# All setup is done in 00-import.ipynb - no additional configuration needed here


## Load NotePlan Files

Get NotePlan files to process. Note: `NOTEPLAN_DIR` is already set up in `00-import.ipynb` based on the runtime environment.


In [3]:
# Get NotePlan files from the last month
files = get_files_from_last_month(NOTEPLAN_DIR)
print(f"Found {len(files)} files to process")

# Filter out files we should skip
files = [(fp, mod_time) for fp, mod_time in files if not should_skip_file(fp)]
print(f"After filtering: {len(files)} files")

# Limit to first 5 files for demo (remove this limit for full processing)
files = files[:5]
print(f"Processing {len(files)} files for this demo")


Found 193 files to process
After filtering: 193 files
Processing 5 files for this demo


## Extract Entities and Relationships

Use the graph builder agent to extract entities and relationships from each note.


In [4]:
# Enable nested event loops for Jupyter notebooks
# This allows asyncio.run() to work even when an event loop is already running
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    print("⚠️  nest_asyncio not installed. Install with: pip install nest-asyncio")
    print("   You may encounter 'asyncio.run() cannot be called from a running event loop' errors")

# Use utility function from knowledge_agents package
from knowledge_agents.utils.graph_utils import extract_from_note_file

# Process all files
extracted_data = []
for file_path, mod_time in files:
    relative_path = str(file_path.relative_to(NOTEPLAN_DIR))
    print(f"Processing: {relative_path}")
    
    # nest_asyncio allows asyncio.run() to work in Jupyter notebooks
    path, output = asyncio.run(extract_from_note_file(file_path, relative_path, dependencies))
    if output:
        extracted_data.append((path, output))
        print(f"  ✅ Extracted {len(output.entities)} entities, {len(output.relationships)} relationships")

print(f"\n✅ Processed {len(extracted_data)} files successfully")


Processing: np-out.log


RuntimeError: asyncio.run() cannot be called from a running event loop

## Display Extracted Data

View the extracted entities and relationships.


In [ ]:
# Collect all entities and relationships
all_entities = []
all_relationships = []

for file_path, output in extracted_data:
    for entity in output.entities:
        all_entities.append({
            "file": file_path,
            "name": entity.name,
            "type": entity.type,
            "properties": entity.properties
        })
    for rel in output.relationships:
        all_relationships.append({
            "file": file_path,
            "from": rel.from_entity,
            "type": rel.type,
            "to": rel.to_entity,
            "properties": rel.properties
        })

# Display entities
if all_entities:
    print("Extracted Entities:")
    df_entities = pd.DataFrame(all_entities)
    print(df_entities.head(20))
    print(f"\nTotal entities: {len(all_entities)}")

# Display relationships
if all_relationships:
    print("\nExtracted Relationships:")
    df_relationships = pd.DataFrame(all_relationships)
    print(df_relationships.head(20))
    print(f"\nTotal relationships: {len(all_relationships)}")


## Next Steps

Now that data is extracted, proceed to:
- [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb): Generate vector embeddings from NotePlan notes
- [**03-loading-data.ipynb**](./03-loading-data.ipynb): Load entities and relationships into Neo4j graph
- [**04o1-loading-vector-embeddings-neo4j.ipynb**](./04o1-loading-vector-embeddings-neo4j.ipynb): Store vector embeddings in Neo4j
